# Module 2: Sentiment and Emotion Classifier

This notebook trains a neural Bidirectional LSTM (BiLSTM) model on the `dair-ai/emotion` dataset, mapping granular emotional categories into three operational tone buckets: Negative (Frustrated), Neutral, and Positive (Satisfied).

In [ ]:
import os
import re
import unicodedata
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

## 1. Load Dataset and Map Emotion Buckets

In [ ]:
dataset_train = load_dataset('dair-ai/emotion', split='train')
dataset_test = load_dataset('dair-ai/emotion', split='test')

emotion_to_sentiment = {
    0: 0,
    1: 2,
    2: 2,
    3: 0,
    4: 0,
    5: 1
}
sentiment_names = {0: 'negative', 1: 'neutral', 2: 'positive'}

def normalize_text(text):
    text = unicodedata.normalize('NFKC', str(text))
    text = text.lower().strip()
    text = re.sub(r'\s+', ' ', text)
    return text

train_texts = [normalize_text(t) for t in dataset_train['text']]
train_labels = [emotion_to_sentiment[l] for l in dataset_train['label']]

test_texts = [normalize_text(t) for t in dataset_test['text']]
test_labels = [emotion_to_sentiment[l] for l in dataset_test['label']]

print('Train samples:', len(train_texts))
print('Test samples:', len(test_texts))

## 2. Vocabulary Construction and PyTorch Dataset

In [ ]:
vocab = {'<pad>': 0, '<unk>': 1}
for text in train_texts:
    for w in text.split():
        if w not in vocab:
            vocab[w] = len(vocab)

print('Vocabulary size:', len(vocab))

class TextDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len=64):
        self.texts = texts
        self.labels = labels
        self.vocab = vocab
        self.max_len = max_len
        
    def __len__(self):
        return len(self.texts)
        
    def __getitem__(self, idx):
        words = self.texts[idx].split()
        indices = [self.vocab.get(w, self.vocab['<unk>']) for w in words][:self.max_len]
        if len(indices) < self.max_len:
            indices = indices + [self.vocab['<pad>']] * (self.max_len - len(indices))
        return torch.tensor(indices, dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.long)

train_loader = DataLoader(TextDataset(train_texts, train_labels, vocab), batch_size=64, shuffle=True)
test_loader = DataLoader(TextDataset(test_texts, test_labels, vocab), batch_size=128, shuffle=False)

## 3. BiLSTM Architecture Definition

In [ ]:
class BiLSTMSentimentNet(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True
        )
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_dim * 2, output_dim)
        
    def forward(self, x):
        embedded = self.embedding(x)
        output, (hidden, cell) = self.lstm(embedded)
        combined = torch.cat((hidden[-2, :, :], hidden[-1, :, :]), dim=1)
        dropped = self.dropout(combined)
        return self.fc(dropped)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = BiLSTMSentimentNet(len(vocab), 64, 64, 3, vocab['<pad>']).to(device)
print(model)

## 4. Model Training and Evaluation

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.003)

epochs = 3
for epoch in range(epochs):
    model.train()
    epoch_loss = 0.0
    for x_batch, y_batch in train_loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(x_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    print(f'Epoch {epoch+1}/{epochs} - Loss: {epoch_loss/len(train_loader):.4f}')

model.eval()
all_preds = []
all_targets = []
with torch.no_grad():
    for x_batch, y_batch in test_loader:
        x_batch = x_batch.to(device)
        outputs = model(x_batch)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_targets.extend(y_batch.numpy())

target_names = ['negative', 'neutral', 'positive']
print(f'Accuracy: {accuracy_score(all_targets, all_preds) * 100:.2f}%\n')
print(classification_report(all_targets, all_preds, target_names=target_names))